# Data preprocessing 
This version works for DyGKT-modified version. It is based on top of the DyGKT preprocessing step and takes inspiration from it. 

Author: Riccardo Alfonso Cerrone

## Personal notes
- It seems that a student may correspond to many student IDs in the pre-processed dataset given by DyGKT. In this Jupyter notebook I am assuming that a student ID corresponds to a single student, and vice-versa.
- The ts time in the pre-processed dataset in DyGKT is the 'endTime' in assistments dataset.
- I changed the way the 'noskill' in assistments dataset has been handled: in my case I am assuming that it has skill = 0, so that we don't have to do the neighbor contextualized sampling in that case (they are not related).
- It seems that the skill of a node is overwritten by the last interaction skill related to the question node we are looking for.

## Code

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

In [10]:
# Define the dataset
dataset = 'assist12'
dataset_name = dataset

In [11]:
# Load the dataset
df_original = pd.read_csv(
    f"../DG_data/{dataset_name}/{dataset_name}.csv",
    encoding="latin1"
)

df_original.columns

Index(['problem_log_id', 'skill', 'problem_id', 'user_id', 'assignment_id',
       'assistment_id', 'start_time', 'end_time', 'problem_type', 'original',
       'correct', 'bottom_hint', 'hint_count', 'actions', 'attempt_count',
       'ms_first_response', 'tutor_mode', 'sequence_id', 'student_class_id',
       'position', 'type', 'base_sequence_id', 'skill_id', 'teacher_id',
       'school_id', 'overlap_time', 'template_id', 'answer_id', 'answer_text',
       'first_action', 'problemlogid', 'Average_confidence(FRUSTRATED)',
       'Average_confidence(CONFUSED)', 'Average_confidence(CONCENTRATING)',
       'Average_confidence(BORED)'],
      dtype='object')

In [12]:
# For assist12
studentColumn = 'user_id'
problemColumn = 'problem_id'
endTime = 'end_time'
skill = 'skill'

# Use the same internal column names for every dataset.
df_original = df_original.rename(columns={
    studentColumn: "studentId",
    problemColumn: "problemId",
    endTime: "endTime",
    skill: "skill",
})

# Convert the problem completion time to UNIX time in seconds.
# The source timestamps are interpreted as UTC.
end_time_text = df_original["endTime"].astype("string").str.strip()
has_fractional_seconds = end_time_text.str.contains(r"\.\d+$", na=False)
normalized_end_time = end_time_text.where(
    has_fractional_seconds,
    end_time_text + ".0",
)
parsed_end_time = pd.to_datetime(
    normalized_end_time,
    format="%Y-%m-%d %H:%M:%S.%f",
    errors="coerce",
    utc=True,
)
if parsed_end_time.isna().any():
    invalid_rows = parsed_end_time[parsed_end_time.isna()].index[:10].tolist()
    raise ValueError(f"Invalid end_time values at rows: {invalid_rows}")

df_original["endTime"] = (
    parsed_end_time.array.asi8 // 1_000_000_000
)
df_original["skill"] = df_original["skill"].fillna("noskill")

assert df_original["endTime"].dtype == np.int64
assert (df_original["endTime"] > 0).all()


In [13]:
df_original.head()
df_original['correct'].value_counts()

correct
1.000    4141564
0.000    1976383
0.500       2395
0.750       1629
0.250       1211
0.600         24
0.975         24
0.850         10
0.650         10
0.950          9
0.875          6
0.625          4
0.375          1
Name: count, dtype: int64

In [14]:
MIN_INTERACTIONS = 5

df = df_original.copy()
previous_num_rows = -1

while previous_num_rows != len(df):
    previous_num_rows = len(df)

    student_counts = df["studentId"].value_counts()
    valid_students = student_counts[
        student_counts >= MIN_INTERACTIONS
    ].index
    df = df[df["studentId"].isin(valid_students)]

    question_counts = df["problemId"].value_counts()
    valid_questions = question_counts[
        question_counts >= MIN_INTERACTIONS
    ].index
    df = df[df["problemId"].isin(valid_questions)]

df = (
    df.sort_values(by="endTime", ascending=True)
      .reset_index(drop=True)
)

print(f"Original interactions: {len(df_original):,}")
print(f"Filtered interactions: {len(df):,}")
print(f"Removed interactions: {len(df_original) - len(df):,}")
print(f"Students: {df['studentId'].nunique():,}")
print(f"Questions: {df['problemId'].nunique():,}")

assert df["studentId"].value_counts().min() >= MIN_INTERACTIONS
assert df["problemId"].value_counts().min() >= MIN_INTERACTIONS

df.head()

Original interactions: 6,123,270
Filtered interactions: 6,001,504
Removed interactions: 121,766
Students: 42,958
Questions: 105,373


,problem_log_id,skill,problemId,studentId,assignment_id,assistment_id,start_time,endTime,problem_type,original,...,overlap_time,template_id,answer_id,answer_text,first_action,problemlogid,Average_confidence(FRUSTRATED),Average_confidence(CONFUSED),Average_confidence(CONCENTRATING),Average_confidence(BORED)
0,136843144,noskill,185963,169036,486567,97308,2012-09-01 00:00:38.696652,1346457646,fill_in_1,0,...,7441,30482,NaN,180,0,136843144,0.361323,0.0,0.766925,0.442968
1,136843145,noskill,185964,169036,486567,97308,2012-09-01 00:00:46.283062,1346457687,algebra,0,...,41608,30482,NaN,.6666666,0,136843145,0.361323,0.0,0.766925,0.442968
2,136843146,Unit Rate,186129,169036,486567,97369,2012-09-01 00:01:31.798664,1346457708,algebra,1,...,16588,46347,NaN,1.26,0,136843146,0.361323,0.0,0.336529,0.762689
3,136843143,Making a Table from an Equation,232301,172027,486577,121637,2012-09-01 00:00:06,1346457726,algebra,1,...,40518,116340,NaN,7,0,136843143,0.361323,0.0,0.766925,0.442968
4,136843147,noskill,390031,174120,548857,222517,2012-09-01 00:01:34,1346457728,choose_1,1,...,34564,222517,NaN,B. 467,0,136843147,0.361323,0.0,0.336529,0.762689


In [15]:
# Defining paths where to save the pre-processed dataset
Path(f"../processed_data/{dataset}").mkdir(parents=True, exist_ok=True)
PATH = '../DG_data/{}/{}.csv'.format(dataset_name, dataset_name)
OUT_DF = '../processed_data/{}/ml_{}.csv'.format(dataset_name, dataset_name)
OUT_FEAT = '../processed_data/{}/ml_{}.npy'.format(dataset_name, dataset_name)
OUT_NODE_FEAT = '../processed_data/{}/ml_{}_node.npy'.format(dataset_name, dataset_name)

In [16]:
# Get student IDs
# Initialize the new student IDs
fromStudentToIDStudent = {}
fromIDStudentToStudent = {}

for i in range(len(df)):
    student = df.loc[i, "studentId"]
    if student not in fromStudentToIDStudent:
        new_id = len(fromStudentToIDStudent)
        fromStudentToIDStudent[student] = new_id
        fromIDStudentToStudent[new_id] = student

In [17]:
# Get skill IDs
fromSkillToIDSkill = {}
fromIDSkillToSkill = {}

# Append the first skill with index 1: 'noskill'
# THIS MUST BE CHANGED ACCORDING TO THE DATASET
no_skill_name = 'noskill'
fromSkillToIDSkill[no_skill_name] = 1
fromIDSkillToSkill[1] = no_skill_name

for i in range(len(df)):
          skill = df.loc[i, "skill"]
          if skill not in fromSkillToIDSkill:
                    new_id = len(fromSkillToIDSkill) + 1
                    fromSkillToIDSkill[skill] = new_id
                    fromIDSkillToSkill[new_id] = skill

In [18]:
# Get question IDs
# Initialize the new question IDs
fromQuestionToIDQuestion = {}
fromIDQuestionToQuestion = {}

# I also want to store the skill associated to it
fromIDQuestionToIDSkill = {}

# Approach that I noticed in the paper -> last skill is mantained (overlap)
# Main assumption is: a question is associated AT MOST to a skill

for i in range(len(df)):
          # Store the new ID of the question
          question = df.loc[i, "problemId"]
          if question not in fromQuestionToIDQuestion:
                    new_id = len(fromQuestionToIDQuestion)
                    fromQuestionToIDQuestion[question] = new_id
                    fromIDQuestionToQuestion[new_id] = question
                    id_node = new_id
          else:
                    id_node = fromQuestionToIDQuestion[question]
          # Storing the skill associated to that question
          skill = df.loc[i, "skill"]
          fromIDQuestionToIDSkill[id_node] = fromSkillToIDSkill[skill]

In [19]:
# What I want in my CSV file:
# user, question, time, skill, label, edge_idx

processed_df_list = []

u_list, i_list, ts_list, skill_list, label_list = [], [], [], [], []
feat_l = []
idx_list = []

# Sort before assigning edge indices
df = df.sort_values("endTime").reset_index(drop=True)

previous_time = -1

for idx, e in enumerate(df.itertuples(index=False), start=1):

    # User ID
    u = fromStudentToIDStudent[e.studentId]

    # Item ID / question ID
    i = fromQuestionToIDQuestion[e.problemId]

    # Timestamp
    ts = np.float64(e.endTime)

    # Check whether timestamps are in ascending order
    assert ts >= previous_time
    previous_time = ts

    # Skill
    skill = e.skill

    # Edge feature / answer correctness: only a fully correct answer is positive.
    # Every score below 1.0 is converted to 0.0.
    label = np.float32(1.0 if e.correct == 1.0 else 0.0)

    # Store values
    u_list.append(u)
    i_list.append(i)
    ts_list.append(ts)
    skill_list.append(skill)
    label_list.append(label)

    # Edge index
    idx_list.append(idx)

    # Edge features must have shape (num_edges, num_features)
    feat_l.append([label])

    processed_df_list.append(
        (u, i, ts, label, idx)
    )

# Output of the pre-process step
processed_df = pd.DataFrame(
    processed_df_list,
    columns=["u", "i", "ts", "label", "idx"]
)  
edge_feats = np.array(feat_l)

In [20]:
processed_df

,u,i,ts,label,idx
0,0,0,1.346458e+09,1.0,1
1,0,1,1.346458e+09,0.0,2
2,0,2,1.346458e+09,1.0,3
3,1,3,1.346458e+09,0.0,4
4,2,4,1.346458e+09,1.0,5
...,...,...,...,...,...
6001499,41411,4194,1.381093e+09,0.0,6001500
6001500,36796,9912,1.381251e+09,0.0,6001501
6001501,36796,12322,1.381253e+09,0.0,6001502
6001502,36796,3318,1.381253e+09,0.0,6001503


In [21]:
bipartite = True
new_df = processed_df.copy()
if bipartite:
          # check the ids of users and items
          assert (processed_df.u.max() - processed_df.u.min() + 1 == len(processed_df.u.unique()))
          assert (processed_df.i.max() - processed_df.i.min() + 1 == len(processed_df.i.unique()))
          assert processed_df.u.min() == processed_df.i.min() == 0

# if bipartite, discriminate the source and target node by unique ids (target node id is counted based on source node id)
upper_u = processed_df.u.max() + 1
new_i = processed_df.i + upper_u

new_df.i = new_i

# make the id start from 1
new_df.u += 1
new_df.i += 1
#new_df.idx += 1

In [22]:
# This code comes from the pre-processing (I want to adapt it)
#df, edge_feats = preprocess(PATH)
#new_df = reindex(df, bipartite)

node_feat_dim = 1
df = new_df

# edge feature for zero index, which is not used (since edge id starts from 1)
empty = np.zeros(edge_feats.shape[1])[np.newaxis, :]
# Stack arrays in sequence vertically (row wise),
edge_feats = np.vstack([empty, edge_feats])

# node features with one additional feature for zero index (since node id starts from 1)
max_idx = max(new_df.u.max(), new_df.i.max())
node_feat_dim = 1

max_idx = max(new_df.u.max(), new_df.i.max())

# Row 0 is reserved for padding
node_feats = np.zeros(
    (max_idx + 1, node_feat_dim),
    dtype=np.float32
)
# Assign the skill to each question node
for question_node_id_raw, skill_id in fromIDQuestionToIDSkill.items():
    # Same offset used above for item IDs, including the final +1 for padding.
    question_node_id = question_node_id_raw + processed_df.u.max() + 2
    node_feats[question_node_id, 0] = skill_id

          
print('number of nodes ', node_feats.shape[0] - 1)
print('number of node features ', node_feats.shape[1])
print('number of edges ', edge_feats.shape[0] - 1)
print('number of edge features ', edge_feats.shape[1])

new_df.to_csv(OUT_DF)  # edge-list
np.save(OUT_FEAT, edge_feats)  # edge features
np.save(OUT_NODE_FEAT, node_feats)  # node features

number of nodes  148331
number of node features  1
number of edges  6001504
number of edge features  1
